# Trajectory inference with bowside

Here, we use bowside for trajectory inference (specifically: cell fate inference) for benchmarking its performance.
We use the synthetic curated datasets from BEELINE ([Pratapa et al. (2020)](https://doi.org/10.1038/s41592-019-0690-6)), downloaded from [Zenodo](https://doi.org/10.5281/zenodo.19009603).

In [1]:
from pathlib import Path

import optuna
import scanpy as sc
import torch
from bowside.dataset import PseudotimeDataset
from bowside.model import GRNModelSDE
from bowside.trajectory import infer_cell_fates, infer_trajectories
from optuna.storages import JournalStorage
from optuna.storages.journal import JournalFileBackend

In [2]:
# Load the data and construct PseudotimeDatasets
dataset_names = ["GSD", "HSC", "mCAD", "VSC"]
adatas = {
    name: [
        sc.read_h5ad(
            f"../data/BEELINE-data/processed/Curated/{name}/{name}_processed_{i + 1}.h5ad"
        )
        for i in range(10)
    ]
    for name in dataset_names
}
datasets = {
    name: [
        PseudotimeDataset(
            adata,
            layer="log1p",
            mask_var=None,
            pseudotime_key="pseudotime",
            num_cells=200,
        )
        for adata in lst
    ]
    for name, lst in adatas.items()
}

In [3]:
# Load tuned bowside hyperparameters (for model initialization)
studies = {
    name: optuna.load_study(
        study_name=f"{name}_tuning_weighted_sinkhorn_time_point_split",
        storage=JournalStorage(
            JournalFileBackend(
                f"../output/tuning/BEELINE_{name}/weighted_sinkhorn_time_point_split/journal.log"
            )
        ),
    )
    for name in dataset_names
}

In [4]:
# Create output directories
out_dirs = {
    name: Path(f"../output/BEELINE_{name}/bowside_optimized") for name in dataset_names
}
for out_dir in out_dirs.values():
    out_dir.mkdir(parents=True, exist_ok=True)

In [5]:
# Run bowside trajectory inference on all datasets
for name, lst in datasets.items():
    params = studies[name].best_params
    for i, dataset in enumerate(lst):
        sigma = params["sigma"]
        adj_init_normal = params["adj_init_normal"] == "True"
        shared_sigma = params["shared_sigma"] == "True"
        fixed_sigma = params["fixed_sigma"] == "True"
        constant_noise = params["constant_noise"] == "True"

        model = GRNModelSDE(
            dataset.n_genes(),
            adj=None,
            adj_init_normal=adj_init_normal,
            activation="sigmoid",
            sigma=sigma,
            shared_sigma=shared_sigma,
            fixed_sigma=fixed_sigma,
            constant_noise=constant_noise,
        )
        checkpoint = torch.load(
            f"../output/BEELINE_{name}/bowside_optimized/model_{i}/checkpoint_final_model.tar",
            weights_only=True,
            map_location=torch.device("cpu"),
        )
        model.load_state_dict(checkpoint["model_state_dict"])

        # Infer trajectories and cell fates
        infer_trajectories(dataset, model, dt=0.01)
        infer_cell_fates(dataset, allow_singular_covariance=True)
        results_df = sc.get.obs_df(
            dataset.adata,
            keys="bowside_fate_entropy",
            obsm_keys=[
                ("bowside_fate_probabilities", col)
                for col in range(dataset.adata.uns["bowside"]["num_trajectories"])
            ],
        )
        results_df.rename(
            columns={
                f"bowside_fate_probabilities-{col}": f"fate_probability_{col}"
                for col in range(dataset.adata.uns["bowside"]["num_trajectories"])
            },
            inplace=True,
        )
        results_df.to_parquet(
            out_dirs[name] / f"model_{i}" / "fate_probabilities.parquet"
        )